# Day 2: Docker & APIs for ML

**MLOps in Practice Workshop**

---

## Learning Objectives

1. Understand Docker concepts: images, containers, and layers
2. Write Dockerfiles for ML applications
3. Build a FastAPI prediction endpoint with Pydantic validation
4. Use Docker Compose for multi-container setups
5. Serve a scikit-learn model via a containerized API

---

## 1. Docker Concepts

Docker solves the "it works on my machine" problem by packaging applications and their dependencies into portable containers.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Image** | A read-only template containing the OS, dependencies, and application code. Built from a Dockerfile. |
| **Container** | A running instance of an image. Isolated, lightweight, and ephemeral. |
| **Layer** | Each instruction in a Dockerfile creates a layer. Layers are cached and shared between images. |
| **Registry** | A storage for images (Docker Hub, AWS ECR, GCP Artifact Registry). |
| **Volume** | Persistent storage that survives container restarts. Used for data and models. |

### How Layers Work

```
+---------------------------+
| Layer 5: COPY app code    |  <-- Changes often (rebuilt frequently)
+---------------------------+
| Layer 4: pip install      |  <-- Changes when requirements change
+---------------------------+
| Layer 3: COPY requirements|  <-- Separated so pip layer is cached
+---------------------------+
| Layer 2: WORKDIR /app     |  <-- Rarely changes
+---------------------------+
| Layer 1: python:3.10-slim |  <-- Base image (cached)
+---------------------------+
```

**Optimization principle:** Put things that change frequently at the bottom of the Dockerfile and things that change rarely at the top. This maximizes layer caching.

---

## 2. Dockerfile Anatomy

Here is a well-structured Dockerfile for an ML prediction service:

```dockerfile
# Use a slim Python base image to reduce size
FROM python:3.10-slim

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Set working directory inside the container
WORKDIR /app

# Install system dependencies (if needed for numpy/scipy)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (layer caching optimization)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code
COPY app.py .
COPY models/ ./models/

# Expose the port the app runs on
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

# Run the application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Key Dockerfile Instructions

| Instruction | Purpose |
|------------|----------|
| `FROM` | Set the base image |
| `ENV` | Set environment variables |
| `WORKDIR` | Set the working directory |
| `COPY` | Copy files from host to container |
| `RUN` | Execute a command during build |
| `EXPOSE` | Document which port the app uses |
| `CMD` | Default command when the container starts |
| `HEALTHCHECK` | Define how Docker checks if the container is healthy |

### Common Docker Commands

```bash
# Build an image
docker build -t ml-api:v1 .

# Run a container
docker run -d -p 8000:8000 --name ml-service ml-api:v1

# View logs
docker logs ml-service

# Stop and remove
docker stop ml-service && docker rm ml-service

# List images and containers
docker images
docker ps -a
```

---

## 3. Building a FastAPI Prediction Endpoint

FastAPI is the go-to framework for ML APIs because it provides:
- Automatic request/response validation via Pydantic
- Auto-generated OpenAPI docs at `/docs`
- Async support for high-throughput serving

### 3.1 Train and Save a Model

In [ ]:
# Install dependencies
# !pip install fastapi uvicorn scikit-learn pandas numpy joblib pydantic

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import json
import os

# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Build a pipeline (scaler + model) so we ship one artifact
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Save model and metadata
os.makedirs("sample_app/models", exist_ok=True)
joblib.dump(pipeline, "sample_app/models/iris_pipeline.joblib")

metadata = {
    "model_type": "RandomForest",
    "feature_names": iris.feature_names.tolist(),
    "target_names": iris.target_names.tolist(),
    "accuracy": float(accuracy_score(y_test, y_pred)),
}
with open("sample_app/models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model and metadata saved to sample_app/models/")

### 3.2 FastAPI Application Code

Below is the full FastAPI app (also available at `sample_app/app.py`):

In [ ]:
# This is the content of sample_app/app.py
# We display it here for reference -- it runs as a standalone server, not in Jupyter.

fastapi_code = '''
import joblib
import json
import numpy as np
from pathlib import Path
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional

# --------------- Pydantic Models ---------------

class PredictionRequest(BaseModel):
    """Input schema for a single prediction."""
    sepal_length: float = Field(..., ge=0, le=10, description="Sepal length in cm")
    sepal_width: float = Field(..., ge=0, le=10, description="Sepal width in cm")
    petal_length: float = Field(..., ge=0, le=10, description="Petal length in cm")
    petal_width: float = Field(..., ge=0, le=10, description="Petal width in cm")

class PredictionResponse(BaseModel):
    """Output schema for predictions."""
    prediction: str
    prediction_id: int
    probabilities: dict

class BatchRequest(BaseModel):
    """Input schema for batch predictions."""
    instances: List[PredictionRequest]

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool
    model_type: Optional[str] = None

# --------------- App Setup ---------------

app = FastAPI(
    title="Iris Prediction API",
    description="Predict iris species from flower measurements.",
    version="1.0.0",
)

# Load model at startup
MODEL_PATH = Path("models/iris_pipeline.joblib")
METADATA_PATH = Path("models/metadata.json")

model = None
metadata = None

@app.on_event("startup")
def load_model():
    global model, metadata
    if MODEL_PATH.exists():
        model = joblib.load(MODEL_PATH)
    if METADATA_PATH.exists():
        with open(METADATA_PATH) as f:
            metadata = json.load(f)

# --------------- Endpoints ---------------

@app.get("/health", response_model=HealthResponse)
def health_check():
    return HealthResponse(
        status="healthy" if model is not None else "unhealthy",
        model_loaded=model is not None,
        model_type=metadata.get("model_type") if metadata else None,
    )

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    features = np.array([
        [request.sepal_length, request.sepal_width,
         request.petal_length, request.petal_width]
    ])

    prediction_id = int(model.predict(features)[0])
    probabilities = model.predict_proba(features)[0]

    target_names = metadata.get("target_names", ["0", "1", "2"])

    return PredictionResponse(
        prediction=target_names[prediction_id],
        prediction_id=prediction_id,
        probabilities={name: float(prob) for name, prob in zip(target_names, probabilities)},
    )

@app.post("/predict/batch", response_model=List[PredictionResponse])
def predict_batch(request: BatchRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    features = np.array([
        [inst.sepal_length, inst.sepal_width, inst.petal_length, inst.petal_width]
        for inst in request.instances
    ])

    predictions = model.predict(features)
    probabilities = model.predict_proba(features)
    target_names = metadata.get("target_names", ["0", "1", "2"])

    return [
        PredictionResponse(
            prediction=target_names[int(pred)],
            prediction_id=int(pred),
            probabilities={name: float(prob) for name, prob in zip(target_names, probs)},
        )
        for pred, probs in zip(predictions, probabilities)
    ]
'''

print(fastapi_code)

### 3.3 Testing the API Locally

To run the API outside Docker:

```bash
cd sample_app
uvicorn app:app --reload --port 8000
```

Then visit `http://localhost:8000/docs` for the interactive Swagger UI.

In [ ]:
# Simulate what an API client would send
import json

# Example single request
single_request = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2,
}

# Example batch request
batch_request = {
    "instances": [
        {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2},
        {"sepal_length": 6.7, "sepal_width": 3.0, "petal_length": 5.2, "petal_width": 2.3},
        {"sepal_length": 5.9, "sepal_width": 3.0, "petal_length": 4.2, "petal_width": 1.5},
    ]
}

print("Single request payload:")
print(json.dumps(single_request, indent=2))

print("\nBatch request payload:")
print(json.dumps(batch_request, indent=2))

print("\nTo call the API with curl:")
print('curl -X POST http://localhost:8000/predict \\')
print('  -H "Content-Type: application/json" \\')
print(f'  -d \'{json.dumps(single_request)}\'')

In [ ]:
# Directly test prediction logic (without running the server)
pipeline = joblib.load("sample_app/models/iris_pipeline.joblib")

test_input = np.array([[5.1, 3.5, 1.4, 0.2]])
prediction = pipeline.predict(test_input)
probabilities = pipeline.predict_proba(test_input)

print(f"Prediction: {iris.target_names[prediction[0]]}")
print(f"Probabilities: {dict(zip(iris.target_names, probabilities[0].round(4)))}")

---

## 4. Docker Compose for Multi-Container Setups

In production, ML systems often require multiple services. Docker Compose orchestrates them.

```yaml
# docker-compose.yml
version: "3.8"

services:
  ml-api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models
    environment:
      - MODEL_PATH=/app/models/iris_pipeline.joblib
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped

  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.10.0
    ports:
      - "5000:5000"
    volumes:
      - mlflow-data:/mlflow
    command: mlflow server --host 0.0.0.0 --port 5000 --backend-store-uri sqlite:///mlflow/mlflow.db --default-artifact-root /mlflow/artifacts

  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml

volumes:
  mlflow-data:
```

### Docker Compose Commands

```bash
# Build and start all services
docker compose up -d --build

# View logs
docker compose logs -f ml-api

# Stop all services
docker compose down

# Remove volumes too
docker compose down -v
```

---

## 5. Lab: Serve a Scikit-Learn Model via FastAPI

### Objective

Build and run a containerized prediction API for a scikit-learn model.

### Steps

1. **Train a model** on a dataset of your choice (the cell above trains on Iris)
2. **Complete the FastAPI app** in `sample_app/app.py` (provided)
3. **Build the Docker image** and run the container
4. **Test the API** using curl or the Swagger UI

### Exercise: Build and Run

In [ ]:
# Step-by-step commands to build and run the API

commands = """
# 1. Navigate to the sample_app directory
cd sample_app

# 2. Build the Docker image
docker build -t iris-api:v1 .

# 3. Run the container
docker run -d -p 8000:8000 --name iris-service iris-api:v1

# 4. Test the health endpoint
curl http://localhost:8000/health

# 5. Make a prediction
curl -X POST http://localhost:8000/predict \\
  -H "Content-Type: application/json" \\
  -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'

# 6. Clean up
docker stop iris-service && docker rm iris-service
"""
print(commands)

### Challenge Extension

Extend the API with the following features:

1. **Add a `/model/info` endpoint** that returns model metadata (type, accuracy, feature names)
2. **Add request logging** -- log every prediction request with a timestamp
3. **Add a `/predict/csv` endpoint** that accepts a CSV file upload and returns batch predictions
4. **Add authentication** with a simple API key header

In [ ]:
# Skeleton for the challenge extensions

challenge_code = '''
from fastapi import Header, UploadFile, File
import csv
import io
from datetime import datetime

# 1. Model info endpoint
@app.get("/model/info")
def model_info():
    if metadata is None:
        raise HTTPException(status_code=503, detail="Metadata not loaded")
    return metadata

# 2. Request logging (add to predict endpoint)
def log_prediction(request, response):
    log_entry = {
        "timestamp": datetime.utcnow().isoformat(),
        "input": request.dict(),
        "output": response.dict(),
    }
    # In production: write to a database or message queue
    print(f"PREDICTION_LOG: {log_entry}")

# 3. CSV upload endpoint
@app.post("/predict/csv")
async def predict_csv(file: UploadFile = File(...)):
    contents = await file.read()
    reader = csv.DictReader(io.StringIO(contents.decode()))
    # TODO: parse rows, predict, return results
    pass

# 4. API key authentication
API_KEY = "your-secret-key"  # In production: use env variable

@app.post("/predict/secure")
def predict_secure(
    request: PredictionRequest,
    x_api_key: str = Header(...),
):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")
    return predict(request)
'''

print(challenge_code)

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Docker images** | Immutable templates built from Dockerfiles; layers are cached |
| **Containers** | Running instances of images; isolated and reproducible |
| **Dockerfile** | Order instructions for maximum layer caching; copy requirements before code |
| **FastAPI** | Pydantic models validate input/output; auto-generates API docs |
| **Docker Compose** | Orchestrate multiple services (API + MLflow + monitoring) |

### Next Session

**Day 3: Testing & CI/CD** -- We will write tests for ML code and automate them with GitHub Actions.